In [ ]:
# Example 1: coloredlogs.install() — force color mode (isatty=True) on a fresh logger.
# Jupyter's stream is not a tty, so auto-detection would legitimately pick the plain
# BasicFormatter; forcing isatty=True exercises the ColoredFormatter path deterministically.
import io
import logging
import coloredlogs

buf = io.StringIO()
logger = logging.getLogger('example1')
logger.handlers.clear()
coloredlogs.install(level='DEBUG', logger=logger, stream=buf, isatty=True)

logger.debug('debug message')
logger.info('info message')
logger.warning('warning message')
logger.error('error message')
logger.critical('critical message')

assert len(logger.handlers) >= 1, 'coloredlogs.install should attach a handler'
fmt_types = [type(h.formatter).__name__ for h in logger.handlers if h.formatter is not None]
assert any('Colored' in t for t in fmt_types), f'expected a ColoredFormatter, got {fmt_types}'
captured = buf.getvalue()
assert '\x1b[' in captured, 'expected ANSI color codes in forced-color output'
assert 'critical message' in captured
print('OK — handler + ColoredFormatter installed (forced color mode)')


In [ ]:
# Example 2: ColoredFormatter — format a LogRecord directly and inspect the output.
import logging
from coloredlogs import ColoredFormatter

formatter = ColoredFormatter(fmt='%(levelname)s %(name)s %(message)s')
record = logging.LogRecord(
    name='example2',
    level=logging.WARNING,
    pathname=__file__ if '__file__' in dir() else 'nb',
    lineno=1,
    msg='hello %s',
    args=('world',),
    exc_info=None,
)
output = formatter.format(record)
print(repr(output))
assert 'hello world' in output
assert 'WARNING' in output
print('OK — ColoredFormatter produced expected text')


In [ ]:
# Example 3: parse_encoded_styles — turn a style spec string into a style dict.
from coloredlogs import parse_encoded_styles, DEFAULT_LEVEL_STYLES, DEFAULT_FIELD_STYLES

styles = parse_encoded_styles('debug=green;info=cyan;warning=yellow;error=red;critical=red,bold')
print(styles)

assert styles['debug'] == {'color': 'green'}
assert styles['info'] == {'color': 'cyan'}
assert styles['critical'].get('color') == 'red'
assert styles['critical'].get('bold') is True

# Defaults are also exposed as plain dicts.
assert isinstance(DEFAULT_LEVEL_STYLES, dict) and 'info' in DEFAULT_LEVEL_STYLES
assert isinstance(DEFAULT_FIELD_STYLES, dict)
print('OK — parse_encoded_styles + defaults')


In [ ]:
# Example 4: level name <-> number conversions + defined-level/alias discovery.
import logging
from coloredlogs import find_defined_levels, find_level_aliases, level_to_number

assert level_to_number('DEBUG') == logging.DEBUG
assert level_to_number('info') == logging.INFO
assert level_to_number(logging.WARNING) == logging.WARNING
assert level_to_number(25) == 25

defined = find_defined_levels()
print(defined)
assert defined['DEBUG'] == logging.DEBUG
assert defined['CRITICAL'] == logging.CRITICAL

aliases = find_level_aliases()
print(aliases)
assert aliases.get('WARN') == 'WARNING' and aliases.get('FATAL') == 'CRITICAL'
print('OK — level name/number conversions + alias discovery')


In [ ]:
# Example 5: install on the root logger, capture output through a StringIO stream, verify message text.
import io
import logging
import coloredlogs

buf = io.StringIO()
root = logging.getLogger('example5_root')
root.handlers.clear()
coloredlogs.install(level='INFO', logger=root, stream=buf, fmt='%(levelname)s %(message)s', isatty=False)

root.info('captured message')
root.warning('another one')

captured = buf.getvalue()
print(repr(captured))
assert 'captured message' in captured
assert 'another one' in captured
assert 'INFO' in captured and 'WARNING' in captured
print('OK — coloredlogs captured to StringIO')
